# 電商資料 EDA 分析
簡單看一下資料

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 讀資料
df = pd.read_csv('ecommerce_data/orders.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['year_month'] = df['order_date'].dt.to_period('M').astype(str)
print(df.shape)
df.head()

In [ ]:
# 看看2024的營收趨勢
df_2024 = df[df['year'] == 2024]
monthly_rev = df_2024.groupby('year_month')['total_amount'].sum().reset_index()
monthly_rev.columns = ['月份', '營收']
fig = px.line(monthly_rev, x='月份', y='營收', title='2024 月營收趨勢')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# 順便算一下月成長率
monthly_rev['成長率'] = monthly_rev['營收'].pct_change() * 100
print("月成長率:")
print(monthly_rev[['月份', '成長率']].dropna().to_string(index=False))

In [ ]:
# 產品類別分析 - 哪個類別賣最好？
cat_stats = df_2024.groupby('category').agg(
    訂單數=('order_id', 'count'),
    總營收=('total_amount', 'sum'),
    平均單價=('unit_price', 'mean'),
    平均滿意度=('customer_satisfaction', 'mean')
).round(2).sort_values('總營收', ascending=False)
print(cat_stats)

# 畫圖
fig = px.bar(cat_stats.reset_index(), x='category', y='總營收',
             color='平均滿意度', title='各類別營收與滿意度',
             color_continuous_scale='RdYlGn')
fig.show()

# 再來看各州的營收
state_rev = df_2024.groupby('state')['total_amount'].sum().reset_index()
state_rev.columns = ['州', '營收']
fig2 = px.bar(state_rev.sort_values('營收', ascending=True), x='營收', y='州',
              orientation='h', title='各州營收排名')
fig2.show()

In [ ]:
# 客戶滿意度分布
fig = px.histogram(df_2024, x='customer_satisfaction', nbins=5,
                   title='客戶滿意度分布', labels={'customer_satisfaction': '滿意度'})
fig.show()

# 配送天數分析
print(f"平均配送天數: {df_2024['delivery_days'].mean():.1f}")
print(f"配送天數中位數: {df_2024['delivery_days'].median():.0f}")
print(f"超過7天的比例: {(df_2024['delivery_days'] > 7).mean()*100:.1f}%")

# 配送天數 vs 滿意度
delivery_sat = df_2024.groupby('delivery_days')['customer_satisfaction'].mean().reset_index()
fig = px.scatter(delivery_sat, x='delivery_days', y='customer_satisfaction',
                 title='配送天數 vs 客戶滿意度', trendline='ols')
fig.show()

# 2023 vs 2024 比較
for year in [2023, 2024]:
    tmp = df[df['year'] == year]
    print(f"\n--- {year} ---")
    print(f"總訂單: {len(tmp)}")
    print(f"總營收: ${tmp['total_amount'].sum():,.2f}")
    print(f"平均訂單金額: ${tmp['total_amount'].mean():,.2f}")
    print(f"平均滿意度: {tmp['customer_satisfaction'].mean():.2f}")